In [ ]:
import pandas as pd
import time

from static_training import train_static
from progressive_training import train_progressive
from mistake_driven_training import train_mistake_driven
from plot3HeatMaps import plot_three_heatmaps


In [ ]:
df = pd.read_csv("data/Weather_Data.csv", parse_dates=["Date"], dayfirst=True )
# Convert RainToday to boolean
df['RainToday'] = df['RainToday'].map({'Yes':1, 'No':0})

wind_cols = ['WindGustDir', 'WindDir9am', 'WindDir3pm']

# One-hot encode categorical wind directions
df = pd.get_dummies(df, columns=wind_cols, drop_first=False)


# Target: Did it rain today? (boolean)
y = df['RainToday']

# Features
X = df.drop(columns=['RainToday', 'Date', 'Rainfall'])

# Check first few rows
df.head()



In [ ]:
# Training window sizes (1, 3, 6, 12, 18, 24 months)
look_back = [30, 180, 365,  545, 730]  
tree_depth = 5 
# Retrain 3 Months
retrain_step = 90
# Look ahead 1 Month 
look_ahead =  [30, 180, 365,  545, 730]
# retrain if RMSE > 2.5
threshold = .3

In [ ]:
results_static = {}
time_static = {}

for lb in look_back:
    results_static[lb] = {}
    time_static[lb] = {}
    for la in look_ahead:
        start_time = time.time()
        avg_rmse = train_static(X, y, look_back=lb, look_ahead= la, depthOfTree=tree_depth)
        elapsed_time = time.time() - start_time

        results_static[lb][la] = avg_rmse
        time_static[lb][la] = elapsed_time

        print(f"[Static] Look-back {lb}, Look-ahead {la} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

In [ ]:
results_progressive = {}
time_progressive = {}

for lb in look_back:
    results_progressive[lb] = {}
    time_progressive[lb] = {}
    for la in look_ahead:
        start_time = time.time()
        avg_rmse = train_progressive(X, y, look_back=lb, look_ahead=la,
                                        retrain_step=retrain_step, depthOfTree=tree_depth)
        elapsed_time = time.time() - start_time

        results_progressive[lb][la] = avg_rmse
        time_progressive[lb][la] = elapsed_time

        print(f"[Progressive] Look-back {lb}, Look-ahead {la} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

In [ ]:
results_mistake = {}
time_mistake = {}

for lb in look_back:
    results_mistake[lb] = {}
    time_mistake[lb] = {}
    for la in look_ahead:
        start_time = time.time()
        avg_rmse = train_mistake_driven(X, y, look_back=lb, look_ahead=la,
                                           depthOfTree=tree_depth, threshold=threshold)
        elapsed_time = time.time() - start_time

        results_mistake[lb][la] = avg_rmse
        time_mistake[lb][la] = elapsed_time

        print(f"[Mistake-Driven] Look-back {lb}, Look-ahead {la} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

In [ ]:
df_static_rmse = pd.DataFrame(results_static).T
df_static_time = pd.DataFrame(time_static).T

df_progressive_rmse = pd.DataFrame(results_progressive).T
df_progressive_time = pd.DataFrame(time_progressive).T

df_mistake_rmse = pd.DataFrame(results_mistake).T
df_mistake_time = pd.DataFrame(time_mistake).T

In [ ]:
plot_three_heatmaps(
    dfs=[df_static_rmse, df_progressive_rmse, df_mistake_rmse],
    titles=["Static RMSE", "Progressive RMSE", "Mistake-Driven RMSE"],
    vmin=min(df_static_rmse.min().min(), df_progressive_rmse.min().min(), df_mistake_rmse.min().min()),
    vmax=max(df_static_rmse.max().max(), df_progressive_rmse.max().max(), df_mistake_rmse.max().max())
)

plot_three_heatmaps(
    dfs=[df_static_time, df_progressive_time, df_mistake_time],
    titles=["Static Time (s)", "Progressive Time (s)", "Mistake-Driven Time (s)"],
    vmin=min(df_static_time.min().min(), df_progressive_time.min().min(), df_mistake_time.min().min()),
    vmax=max(df_static_time.max().max(), df_progressive_time.max().max(), df_mistake_time.max().max())
)

# Conclusion
When looking out the farthest, progressive and mistake driven training. However if computation is a concern the processing then the smaller windows takes a longer toll due to the increased processing.